# SST DRVI factors — exploration

What does each factor *mean*? Whether the fit is trustworthy is settled in `01_qc.ipynb`;
this notebook assumes it is and goes after interpretation: per-factor activity, the gene
programs each direction encodes, and which factors are only half informative.

DRVI only — scVI's dimensions are not claimed to be individually interpretable, so there
are no traversal scores to read for it.

CPU-only. Kernel: `sst-drvi`.

In [ ]:
import json
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import drvi
import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4))
pd.set_option("display.max_rows", 120)

## Helpers used more than once

`factor_association` is run twice in section 4 — once signed, once split by direction —
and `split_by_sign` is used both directly and from inside it. Everything else this
notebook computes is written out at the point of use.

In [ ]:
# `factor_association` is called twice below and `split_by_sign` twice (once directly,
# once from inside the directional branch), so they are named here; the rest of this
# notebook's statistics are written out where they are used. `logger` is wired up because
# the association code reports which dimensions it drops.
import logging

logger = logging.getLogger("explore")
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
    logger.setLevel(logging.INFO)


def split_by_sign(embed, hide_vanished: bool = True) -> pd.DataFrame:
    """Per-cell activation magnitude of each factor *direction*.

    DRVI factors are directional: ``DR n+`` and ``DR n-`` can encode unrelated programs,
    which is why the interpretability scores are computed per direction. A signed factor
    value conflates the two, so an association carried by only one direction is diluted
    by the cells sitting on the other side of zero. This returns ``relu(x)`` and
    ``relu(-x)`` as separate non-negative columns.

    Columns are labelled and ordered exactly like `interpretability_scores`
    (``DR 1+``, ``DR 1-``, ``DR 2+``, ...), so the two tables join on them. Directions
    flagged vanished by ``set_latent_dimension_stats`` are dropped by default.
    """
    x = np.asarray(embed.X, dtype=np.float32)
    info = embed.var
    columns = {}
    for pos, (_, row) in enumerate(info.iterrows()):
        values = x[:, pos]
        for direction, magnitude, dead in (
            ("+", np.maximum(values, 0.0), row["vanished_positive_direction"]),
            ("-", np.maximum(-values, 0.0), row["vanished_negative_direction"]),
        ):
            if hide_vanished and dead:
                continue
            columns[f"{row['title']}{direction}"] = magnitude

    out = pd.DataFrame(columns, index=embed.obs_names)
    order = (
        pd.concat([info.assign(direction="+"), info.assign(direction="-")])
        .assign(title=lambda df: df["title"] + df["direction"])
        .sort_values(["order", "direction"])["title"]
    )
    return out[[c for c in order if c in out.columns]]


def eta_squared_many(x: np.ndarray, codes: np.ndarray, n_groups: int) -> np.ndarray:
    """Eta-squared of every column of ``x`` against one integer-coded grouping.

    Vectorized with ``bincount``: a per-group Python loop is O(n x n_groups), which is
    unusable at 902 ``library_prep`` levels.
    """
    counts = np.bincount(codes, minlength=n_groups).astype(np.float64)
    live = counts > 0
    grand = x.mean(axis=0)
    total = ((x - grand) ** 2).sum(axis=0)

    out = np.full(x.shape[1], np.nan)
    if live.sum() < 2:
        return out
    for i in range(x.shape[1]):
        if total[i] == 0:
            continue
        sums = np.bincount(codes, weights=x[:, i], minlength=n_groups)
        means = sums[live] / counts[live]
        out[i] = (counts[live] * (means - grand[i]) ** 2).sum() / total[i]
    return out


def associate(
    x: np.ndarray,
    names,
    obs: pd.DataFrame,
    categorical: tuple[str, ...],
    continuous: tuple[str, ...],
) -> pd.DataFrame:
    """Association of every column of ``x`` with each named covariate.

    Categorical -> eta-squared (variance explained); continuous -> ``|Spearman rho|``.
    """
    from scipy import stats

    x = np.asarray(x, dtype=np.float64)
    out = pd.DataFrame(index=pd.Index(names), dtype=float)

    for col in categorical:
        if col not in obs:
            logger.warning("skipping missing obs column %s", col)
            continue
        cat = pd.Categorical(obs[col])
        keep = cat.codes >= 0
        out[col] = eta_squared_many(
            x[keep], cat.codes[keep].astype(np.intp), len(cat.categories)
        )

    for col in continuous:
        if col not in obs:
            logger.warning("skipping missing obs column %s", col)
            continue
        values = pd.to_numeric(obs[col], errors="coerce").to_numpy(dtype=float)
        keep = np.isfinite(values)
        if keep.sum() < 3:
            out[col] = np.nan
            continue
        # ranks every column against the covariate in one call
        rho = stats.spearmanr(x[keep], values[keep]).statistic
        out[col] = np.abs(np.atleast_2d(rho)[:-1, -1]) if x.shape[1] > 1 else abs(rho)
    return out


def factor_association(
    embed,
    categorical: tuple[str, ...] = (),
    continuous: tuple[str, ...] = (),
    directional: bool = False,
) -> pd.DataFrame:
    """Factors x covariates association matrix.

    Categorical covariates get eta-squared (variance explained, 0-1); continuous ones
    get ``|Spearman rho|``, so a single heatmap separates factors driven by biology
    from those tracking technical covariates.

    Parameters
    ----------
    directional
        ``False`` (default) associates on the signed factor value, one row per factor.
        ``True`` splits each factor into its two directions via `split_by_sign`
        first, giving one row per ``DR n+`` / ``DR n-``. Prefer ``True`` whenever a
        factor might be bidirectional -- read it alongside
        `direction_activity`.

    Only non-vanished factors (or directions, when ``directional``) are returned.
    """
    if directional:
        split = split_by_sign(embed, hide_vanished=True)
        return associate(
            split.to_numpy(), split.columns, embed.obs, categorical, continuous
        )

    if {"order", "vanished", "title"} <= set(embed.var.columns):
        used = embed.var.sort_values("order")
        used = used[~used["vanished"]]
        x = np.asarray(embed[:, used.index].X)
        names = used["title"].to_numpy()
    else:
        # An scVI embedding has no vanished/order bookkeeping to reorder by, but it does
        # carry `used` from latent_stats -- and collapsed dimensions MUST be dropped here.
        # Eta-squared is a variance ratio, so it is scale-invariant and cannot tell a live
        # dimension from one sitting at the prior: a dimension with std 0.008 that wiggles
        # slightly with library scores just as high as a real one, which fills the heatmap
        # with noise that reads as signal.
        keep = embed.var.index
        if "used" in embed.var:
            keep = embed.var.index[embed.var["used"].to_numpy().astype(bool)]
            dropped = embed.n_vars - len(keep)
            if dropped:
                logger.info(
                    "dropping %d of %d collapsed latent dimensions (var['used'] is False); "
                    "eta-squared is scale-invariant and would score them like live ones",
                    dropped,
                    embed.n_vars,
                )
        subset = embed[:, keep]
        x = np.asarray(subset.X)
        names = (
            subset.var["title"].to_numpy()
            if "title" in subset.var
            else subset.var_names.to_numpy()
        )
    return associate(x, names, embed.obs, categorical, continuous)

## The fit being interpreted

Prefers the converged covariate refit (`sst_k64_covar_long`), then the 200-epoch covariate
fit, then the baseline — whichever is on disk first. Set `RUN` directly to override.

The order matters: the 200-epoch fits never converged (their validation ELBO is still
falling at ~1.4 nats/epoch on the last epoch — see section 3 of `01_qc.ipynb`), and DRVI's
default `kl_warmup_epochs="auto"` ramps the KL term over the whole run, so a factor that
looks vanished in one of those may simply not have finished forming.

In [ ]:
CANDIDATES = ["sst_k64_covar_long", "sst_k64_covar", "sst_k64"]
MODEL = "drvi"

RUN = None
for candidate in CANDIDATES:
    embed_path = S.EMBED_DIR / f"{candidate}_{MODEL}_embed.h5ad"
    if embed_path.exists():
        RUN = candidate
        break
    print(f"{candidate} not on disk")
if RUN is None:
    raise FileNotFoundError(f"no DRVI embedding found among {CANDIDATES}")

embed = ad.read_h5ad(embed_path)
gene_names = np.asarray(embed.uns["gene_names"]).astype(str)

print(f"run: {RUN}\n{embed}")
print("\nconfig:", json.dumps(json.loads(embed.uns["drvi_config"]), indent=2))
print("\nrun:   ", dict(embed.uns["drvi_run"]))
print("timings:", dict(embed.uns["timings"]))

## 1. Per-factor activity on the UMAP

One panel per non-vanished factor, ordered by reconstruction effect. Factors that light up
a single compact region are the interpretable ones.

In [ ]:
drvi.utils.pl.plot_latent_dims_in_umap(embed, remove_vanished=True, max_cells_to_plot=50_000)

## 2. Factor x group heatmaps

Mean factor value per group, on a balanced subsample so that large supertypes and regions
do not dominate.

In [ ]:
# g, ax = plt.subplots(1,1)
drvi.utils.pl.plot_latent_dims_in_heatmap(embed, "Supertype", remove_vanished=True, figsize=(12,6))

In [ ]:
drvi.utils.pl.plot_latent_dims_in_heatmap(embed, "Brain Region", remove_vanished=True, figsize=(12,6))

## 3. What genes does each factor encode?

The runner stored the out-of-distribution traversal scores in `embed.varm`;
`S.interpretability_scores` rebuilds the genes x factors table from them, so no model
reload (and no GPU) is needed here. `OOD_combined` is DRVI's recommended default;
`IND_max` scores the same factors within the observed data.

In [ ]:
# DRVI's get_interpretability_scores, reproduced from the artifacts `train_drvi.py`
# already wrote so that inspection needs no GPU. Directions are ordered by reconstruction
# effect and titled `DR n+` / `DR n-`, matching the sign-split table in section 4;
# directions flagged vanished by set_latent_dimension_stats are dropped.
effect = np.concatenate([embed.varm["OOD_combined_positive"],
                         embed.varm["OOD_combined_negative"]])
info = (pd.concat([embed.var.assign(direction="+"), embed.var.assign(direction="-")])
        .assign(title=lambda df: df["title"] + df["direction"])
        .reset_index(drop=True))
vanished = np.where(info["direction"] == "+", info["vanished_positive_direction"],
                    info["vanished_negative_direction"])
ordered = info[~vanished].sort_values(["order", "direction"])["title"]

scores = pd.DataFrame(effect, columns=pd.Index(gene_names),
                      index=info["title"]).loc[ordered].T
print(f"{scores.shape[0]} genes x {scores.shape[1]} factor directions")
scores.iloc[:5, :8]

In [ ]:
top = pd.DataFrame([
    {"factor": factor, "rank": rank, "gene": gene, "score": score}
    for factor in scores.columns
    for rank, (gene, score) in enumerate(scores[factor].nlargest(10).items(), start=1)
])
top.pivot(index="rank", columns="factor", values="gene").iloc[:, :12]

In [ ]:
drvi.utils.pl.plot_interpretability_scores(scores, n_top_genes=10, ncols=4, score_threshold=0.1)

### Expression of the top genes of a chosen factor

Swap `factor` for anything from `scores.columns` to sanity-check that its top genes really do
vary with the factor. Counts are read from the prepared cohort for a random 20k-cell subsample
and normalized on the fly (the cohort stores raw counts only).

In [ ]:
factor = scores.columns[0]
factor_genes = scores[factor].nlargest(8).index.tolist()
base_title = factor.rstrip("+-")
dim = embed.var.index[embed.var["title"] == base_title][0]
print(factor, "->", factor_genes)

rng = np.random.default_rng(0)
sub_idx = np.sort(rng.choice(embed.n_obs, size=min(20_000, embed.n_obs), replace=False))

# The cohort stores raw counts only, so normalize the subsample here.
prepared = S.load_prepared(backed="r")
sub = prepared[embed.obs_names[sub_idx]].to_memory()[:, factor_genes].copy()
sc.pp.normalize_total(sub, target_sum=1e4)
sc.pp.log1p(sub)

panel = embed[sub_idx].copy()
panel.obs["factor"] = np.asarray(panel[:, dim].X).ravel()
for gene in factor_genes:
    panel.obs[gene] = np.asarray(sub[:, gene].X.todense()).ravel()

sc.pl.umap(panel, color=["factor"] + factor_genes, ncols=3, size=6, cmap="viridis")

## 4. Split each factor by sign, then re-test

A DRVI factor is directional — `DR n+` and `DR n-` are decoded as separate programs, and
`set_latent_dimension_stats` even flags them vanished independently. Associating on the
*signed* value therefore mixes two programs into one number: if a covariate drives only
the negative direction, the cells sitting above zero dilute it.

So split each factor into `relu(x)` and `relu(-x)` — two non-negative magnitude columns —
and re-run the same association test. Column labels match section 3's score table, so the
two can be read side by side.

In [ ]:
split = split_by_sign(embed)
print(f"{split.shape[1]} live directions from {embed.n_vars} factors")

# How often each direction is active, and how strongly. This is the context the
# association table needs: a direction active in 2% of cells with a high eta-squared
# means something very different from one active in 60%.
activity = pd.DataFrame({
    "frac_active": (split > 0).mean(),
    "mean_when_active": split.where(split > 0).mean(),
    "max": split.max(),
})
print("\nHow often is each direction active? (median frac_active tells you whether "
      "factors are genuinely two-sided)")
print(activity["frac_active"].describe()[["min", "25%", "50%", "75%", "max"]].round(3).to_string())
activity.head(10).round(3)

### Split-direction heatmaps by group

Same style as section 2 (`drvi.utils.pl.plot_latent_dims_in_heatmap` / `sc.pl.heatmap`),
applied to the relu-magnitude split directions. Rows are hierarchically clustered per
heatmap; columns share one order (fixed from the Supertype call) and the two panels are
stacked with a common width so the columns line up vertically between them. Values are
relu magnitudes and never negative, so the colorbar starts at 0 instead of the divergent
default.

In [ ]:
def _split_heatmap(embed, split, groupby, strip_suffix=None, figsize=None, dim_order=None):
    """Factor x group heatmap for split directions, matching the style of section 2.

    Builds a minimal AnnData from the relu-magnitude split DataFrame, runs
    sc.tl.dendrogram to cluster groups by mean direction profile, then calls
    drvi.utils.pl.plot_latent_dims_in_heatmap. Values are relu magnitudes and never
    negative, so the colorbar is vmin=0 with a sequential cmap instead of the
    divergent default.

    Parameters
    ----------
    strip_suffix : str or None
        Remove this suffix from every group label before plotting.
    figsize : tuple or None
        Auto-sized from the number of groups and directions if not given.
    dim_order : array-like or None
        Column positions (into `split.columns`) fixing the direction order. If
        None, columns are sorted by which cell most activates them and the order
        is returned, so a second call can pass it back in — this is what keeps
        the Supertype and Brain Region heatmaps column-aligned.

    Returns
    -------
    dim_order : np.ndarray
    fig : matplotlib.figure.Figure
    """
    # Build minimal AnnData: cells × directions, same obs as embed
    split_ad = ad.AnnData(
        X=split.values.astype(np.float32),
        obs=embed.obs[[groupby]].copy(),
    )
    split_ad.var_names = split.columns
    split_ad.var["vanished"] = False   # already filtered by split_by_sign

    if strip_suffix:
        split_ad.obs[groupby] = (
            split_ad.obs[groupby].astype(str)
            .str.replace(strip_suffix, "", regex=False)
            .astype("category")
        )

    if dim_order is None:
        dim_order = np.abs(split_ad.X).argmax(axis=0).argsort()
    split_ad = split_ad[:, dim_order].copy()
    split_ad.var["title"] = split_ad.var_names.tolist()
    split_ad.var["order"] = np.arange(split_ad.n_vars)

    # Cluster groups hierarchically on their mean direction profiles
    sc.tl.dendrogram(split_ad, groupby)

    if figsize is None:
        n_groups = split_ad.obs[groupby].nunique()
        n_dirs   = split_ad.n_vars
        figsize  = (max(12, n_dirs * 0.22 + 2), max(4, n_groups * 0.22 + 1.5))

    axes = drvi.utils.pl.plot_latent_dims_in_heatmap(
        split_ad, groupby,
        remove_vanished=False,       # already filtered
        sort_by_categorical=False,   # order fixed above so both groupings align
        order_col="order",
        figsize=figsize,
        dendrogram=True,             # show row dendrogram
        cmap="rocket_r", vmin=0, vcenter=None,  # relu magnitudes are never negative
        show=False,
    )
    axes["heatmap_ax"].set_title(groupby)
    return dim_order, axes["heatmap_ax"].figure


# Order is fixed on the Supertype call (finer-grained) and reused for Brain Region
# so the two heatmaps are column-aligned.
dim_order, fig_supertype = _split_heatmap(embed, split, "Supertype", strip_suffix="-SEAAD")
_, fig_region = _split_heatmap(embed, split, "Brain Region", dim_order=dim_order)

# Stack rather than place side by side: a shared figure width plus aspect="auto"
# forces both rasters to span the same x-extent, so the (already column-order-matched)
# factors line up vertically between the two panels.
panel, axs = plt.subplots(
    2, 1,
    figsize=(
        max(fig_supertype.get_size_inches()[0], fig_region.get_size_inches()[0]),
        fig_supertype.get_size_inches()[1] + fig_region.get_size_inches()[1],
    ),
    gridspec_kw={
        "height_ratios": [fig_supertype.get_size_inches()[1], fig_region.get_size_inches()[1]],
    },
)
for ax, fig in zip(axs, (fig_supertype, fig_region)):
    fig.canvas.draw()
    ax.imshow(np.asarray(fig.canvas.buffer_rgba()), aspect="auto")
    ax.axis("off")
    plt.close(fig)
plt.tight_layout()
plt.show()

The signed association is also shown in `01_qc.ipynb`; it is recomputed here
because the comparison below needs both halves in one place.

In [ ]:
CATEGORICAL = ("Supertype", "Brain Region", "Donor ID", "library_prep", "method",
               "Chemistry", "Sex", "Cognitive Status", "Overall AD neuropathological Change",
               "Braak")
CONTINUOUS = ("CPS_Local", "CPS_Global", "CPS_Local_pTau", "CPS_Local_ABeta",
              "Age at Death", "PMI", "Number of UMIs", "Genes detected",
              "Fraction mitochondrial UMIs", "Doublet score")

assoc = factor_association(embed, categorical=CATEGORICAL, continuous=CONTINUOUS)
assoc_dir = factor_association(embed, categorical=CATEGORICAL, continuous=CONTINUOUS,
                                 directional=True)
assoc_dir.round(3)

### Did splitting help?

Per covariate, the strongest association before and after. Symmetric, monotone effects lose a
little (relu throws away half the variance); anything carried by one direction gains.

In [ ]:
comparison = pd.DataFrame({
    "signed_max": assoc.max().round(3),
    "sign_split_max": assoc_dir.max().round(3),
})
comparison["delta"] = (comparison["sign_split_max"] - comparison["signed_max"]).round(3)
comparison["best_direction"] = assoc_dir.idxmax()
comparison.sort_values("sign_split_max", ascending=False)

### Which factors are one-sided?

The gap `|assoc(+) - assoc(-)|` per factor. A large gap means the two halves of one latent
dimension track different things — so the factor is only half informative, and downstream
work should use the split columns rather than the raw signed value.

In [ ]:
# Per-factor gap between the `+` and `-` association profiles. A large gap means the two
# directions of one factor track different covariates, which is exactly the structure a
# signed association test hides.
sides = {}
for name in assoc_dir.index:
    sides.setdefault(name[:-1].strip(), {})[name[-1]] = assoc_dir.loc[name]
asym = pd.DataFrame({
    base: (directions["+"] - directions["-"]).abs()
    for base, directions in sides.items() if set(directions) == {"+", "-"}
}).T
asym.index.name = "factor"

worst = asym.max(axis=1).sort_values(ascending=False)
print("largest +/- disagreement per factor:")
print(worst.head(12).round(3).to_string())

In [ ]:
# Drill into the most asymmetric factors: what drives each side?
for f in worst.head(6).index:
    sides = pd.DataFrame({"+": assoc_dir.loc[f"{f}+"], "-": assoc_dir.loc[f"{f}-"]})
    sides["gap"] = sides["+"] - sides["-"]
    top4 = sides.reindex(sides["gap"].abs().sort_values(ascending=False).index).head(4)
    print(f"\n{f}:")
    print(top4.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(40,8))
sns.heatmap(assoc_dir.T, cmap="rocket_r", vmin=0, vmax=1, linewidths=0.2,cbar_kws={"label": "eta$^2$ / |Spearman $\\rho$|"}, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("factor direction")
ax.set_title("Factor-covariate association, split by sign")
fig.tight_layout()

## 5. Summary

Fill in after looking at the plots above:

- Factors tracking supertype identity:
- Factors tracking brain region:
- Factors tracking neuropathology (`CPS_*`, Braak, AD change):
- Factors that still look technical despite the covariate model:
- One-sided factors from section 4, where only `+` or only `-` is usable:

In [ ]:
from session_info2 import session_info

session_info(dependencies=True)